# 06 Forecasting
Forecast engagement with Prophet when available; fallback to Exponential Smoothing.

In [1]:
import warnings
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

warnings.filterwarnings("ignore")
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "utils" / "utils.py").exists():
    if (PROJECT_ROOT / "notebooks" / "utils" / "utils.py").exists():
        PROJECT_ROOT = PROJECT_ROOT / "notebooks"
    elif (PROJECT_ROOT.parent / "utils" / "utils.py").exists():
        PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from utils.utils import ensure_project_dirs, load_raw_dataset, clean_dataset, PROCESSED_DIR, REPORTS_DIR, FIGURES_DIR
from utils.features import engineer_kpis, build_post_feature_sets, aggregate_business_features
from utils.evaluation import regression_metrics, rank_models
from utils.visualization import set_plot_style, save_figure

set_plot_style()
ensure_project_dirs()
RAW_DATA_PATH = Path("../jsons/all_final_appended.json")
if not RAW_DATA_PATH.exists():
    RAW_DATA_PATH = PROJECT_ROOT / "synthetic_generator" / "synthetic_social_media_posts.csv"
KPI_PATH = PROCESSED_DIR / "kpi_dataset.csv"

## Load KPI and Build Weekly/Monthly Time Series

In [2]:
if KPI_PATH.exists():
    df = pd.read_csv(KPI_PATH, parse_dates=["post_date"])
else:
    df = engineer_kpis(clean_dataset(load_raw_dataset(RAW_DATA_PATH)))
df.head()

,business_name,sector,followers_count,post_date,posting_hour,day_of_week,month,post_type,caption_text,caption_length,...,is_reel,posting_time_bin,caption_length_bin,hashtags_count_bin,emoji_count_bin,engagement_level,business_size_bin,high_engagement,high_view_rate,high_comment_rate
0,LOFT Palestine,Fashion,4392.0,2026-03-31,0.0,Tuesday,3,reel,إطلالة أنثوية بلمسة عصرية ✨ من LOFT. للطلب الت...,88,...,1,night,medium,none,few,medium,small,0,0,0
1,LOFT Palestine,Fashion,4392.0,2026-03-18,0.0,Wednesday,3,reel,اختيارك المثالي لإطلالة كاجول مميزة 🤍✨ LOFT كو...,111,...,1,night,medium,none,few,high,small,1,1,0
2,LOFT Palestine,Fashion,4392.0,2026-03-16,0.0,Monday,3,reel,ستايل شبابي بلمسة عصرية له ولها ✨ اختيارات ممي...,133,...,1,night,medium,none,few,high,small,1,1,1
3,LOFT Palestine,Fashion,4392.0,2026-03-07,0.0,Saturday,3,reel,ستايل يناسبك، وتجربة تستحق الزيارة ✨ زورونا في...,82,...,1,night,medium,none,few,high,small,1,1,0
4,LOFT Palestine,Fashion,4392.0,2026-03-05,0.0,Thursday,3,reel,#LOFTSTYLE #Menstyle,21,...,1,night,short,few,none,high,small,1,1,0


In [3]:
import importlib.util

def build_ts(frame, freq):
    ts = (
        frame.set_index("post_date")["engagement"]
        .resample(freq)
        .mean()
        .dropna()
        .reset_index()
        .rename(columns={"post_date": "ds", "engagement": "y"})
    )
    ts["ds"] = pd.to_datetime(ts["ds"])
    return ts

weekly = build_ts(df, "W-MON")
monthly = build_ts(df, "MS")


## Experiment Grid and Metrics

In [ ]:
has_prophet = importlib.util.find_spec("prophet") is not None
if has_prophet:
    from prophet import Prophet
else:
    from statsmodels.tsa.holtwinters import ExponentialSmoothing
# Forecasting Hyperparameter Experimentation:
# Different forecasting settings are tested to evaluate which configuration produces better prediction performance.
# The selected model is based on evaluation metrics such as MAE/RMSE rather than manual guessing.
cp_vals, modes = [0.01,0.05,0.1,0.5], ["additive","multiplicative"]
rows, store = [], {}
for agg_name, ts in {"weekly": weekly, "monthly": monthly}.items():
    ts = ts.sort_values("ds").reset_index(drop=True)
    if len(ts) < 8:
        continue
    split = int(len(ts) * 0.8)
    train, test = ts.iloc[:split], ts.iloc[split:]
    if len(test) == 0:
        continue
    if has_prophet:
        for cp in cp_vals:
            for mode in modes:
                m = Prophet(changepoint_prior_scale=cp, seasonality_mode=mode, yearly_seasonality=False, weekly_seasonality=True, daily_seasonality=False)
                m.fit(train)
                fut = m.make_future_dataframe(periods=len(test), freq="W-MON" if agg_name=="weekly" else "MS")
                pred = m.predict(fut)[["ds","yhat"]].tail(len(test))
                mets = regression_metrics(test["y"], pred["yhat"])
                rows.append({"model":"prophet","aggregation":agg_name,"changepoint_prior_scale":cp,"seasonality_mode":mode, **mets})
                store[("prophet",agg_name,cp,mode)] = (train,test,pred)
    else:
        model = ExponentialSmoothing(train["y"], trend="add", seasonal=None).fit(optimized=True)
        pred = pd.DataFrame({"ds": test["ds"].values, "yhat": model.forecast(len(test)).values})
        mets = regression_metrics(test["y"], pred["yhat"])
        for cp in cp_vals:
            for mode in modes:
                rows.append({"model":"exp_smoothing_fallback","aggregation":agg_name,"changepoint_prior_scale":cp,"seasonality_mode":mode, **mets})
                store[("exp_smoothing_fallback",agg_name,cp,mode)] = (train,test,pred)

exp_input = pd.DataFrame(rows)
if exp_input.empty:
    exp = pd.DataFrame(columns=["model","aggregation","changepoint_prior_scale","seasonality_mode","MAE","RMSE","MAPE","composite_score"])
    best = None
    forecast = pd.DataFrame(columns=["ds","y","yhat","level","entity"])
else:
    exp = rank_models(exp_input, lower_is_better_cols=["MAE","RMSE","MAPE"])
    best = exp.iloc[0]
    train, test, pred = store[(best["model"],best["aggregation"],best["changepoint_prior_scale"],best["seasonality_mode"])]
    forecast = test.merge(pred, on="ds", how="left")
    forecast["level"], forecast["entity"] = "overall", "all_businesses"


c:\Users\hanai\marketing\notebooks\.venv\Lib\site-packages\statsmodels\tsa\holtwinters\model.py:918: ConvergenceWarning: Optimization failed to converge. Check mle_retvals.
  warnings.warn(
c:\Users\hanai\marketing\notebooks\.venv\Lib\site-packages\statsmodels\tsa\holtwinters\model.py:918: ConvergenceWarning: Optimization failed to converge. Check mle_retvals.
  warnings.warn(


## Sector/Business Extensions and Save

In [5]:
extra = []
if best is not None:
    freq = "W-MON" if best["aggregation"] == "weekly" else "MS"

    def build_segment_ts(seg, freq):
        ts = (
            seg.set_index("post_date")["engagement"]
            .resample(freq)
            .mean()
            .dropna()
            .reset_index()
            .rename(columns={"post_date": "ds", "engagement": "y"})
        )
        ts["ds"] = pd.to_datetime(ts["ds"])
        return ts

    for sector, seg in df.groupby("sector"):
        ts = build_segment_ts(seg, freq)
        if len(ts) < 10:
            continue
        split = int(len(ts)*0.8)
        train_s, test_s = ts.iloc[:split], ts.iloc[split:]
        if len(test_s) < 2:
            continue
        if best["model"] == "prophet":
            from prophet import Prophet
            m = Prophet(changepoint_prior_scale=float(best["changepoint_prior_scale"]), seasonality_mode=best["seasonality_mode"], yearly_seasonality=False, weekly_seasonality=True, daily_seasonality=False)
            m.fit(train_s)
            fut = m.make_future_dataframe(periods=len(test_s), freq=freq)
            pred_s = m.predict(fut)[["ds","yhat"]].tail(len(test_s))
        else:
            from statsmodels.tsa.holtwinters import ExponentialSmoothing
            fit = ExponentialSmoothing(train_s["y"], trend="add", seasonal=None).fit(optimized=True)
            pred_s = pd.DataFrame({"ds": test_s["ds"].values, "yhat": fit.forecast(len(test_s)).values})
        mrg = test_s.merge(pred_s, on="ds", how="left")
        mrg["level"], mrg["entity"] = "sector", sector
        extra.append(mrg)

    for business, seg in df.groupby("business_name"):
        ts = build_segment_ts(seg, freq)
        if len(ts) < 12:
            continue
        split = int(len(ts)*0.8)
        train_b, test_b = ts.iloc[:split], ts.iloc[split:]
        if len(test_b) < 2:
            continue
        if best["model"] == "prophet":
            from prophet import Prophet
            m = Prophet(changepoint_prior_scale=float(best["changepoint_prior_scale"]), seasonality_mode=best["seasonality_mode"], yearly_seasonality=False, weekly_seasonality=True, daily_seasonality=False)
            m.fit(train_b)
            fut = m.make_future_dataframe(periods=len(test_b), freq=freq)
            pred_b = m.predict(fut)[["ds","yhat"]].tail(len(test_b))
        else:
            from statsmodels.tsa.holtwinters import ExponentialSmoothing
            fit = ExponentialSmoothing(train_b["y"], trend="add", seasonal=None).fit(optimized=True)
            pred_b = pd.DataFrame({"ds": test_b["ds"].values, "yhat": fit.forecast(len(test_b)).values})
        mrg = test_b.merge(pred_b, on="ds", how="left")
        mrg["level"], mrg["entity"] = "business", business
        extra.append(mrg)

forecast = pd.concat([forecast] + extra, ignore_index=True) if extra else forecast
forecast.to_csv(PROCESSED_DIR / "forecast.csv", index=False)
exp.to_csv(PROCESSED_DIR / "forecast_metrics.csv", index=False)
display(exp.head(10))
display(forecast.head(20))
if best is None:
    print("No valid forecasting experiments were generated from available data.")
else:
    print("Insight: selected model balances low error and stable behavior across aggregation levels.")


,model,aggregation,changepoint_prior_scale,seasonality_mode,MAE,RMSE,MAPE,composite_score
0,exp_smoothing_fallback,monthly,0.50,multiplicative,14372.100115,21848.206915,2633.892144,2.0
1,exp_smoothing_fallback,monthly,0.50,additive,14372.100115,21848.206915,2633.892144,2.0
2,exp_smoothing_fallback,monthly,0.10,multiplicative,14372.100115,21848.206915,2633.892144,2.0
3,exp_smoothing_fallback,monthly,0.10,additive,14372.100115,21848.206915,2633.892144,2.0
4,exp_smoothing_fallback,monthly,0.05,multiplicative,14372.100115,21848.206915,2633.892144,2.0
5,exp_smoothing_fallback,monthly,0.05,additive,14372.100115,21848.206915,2633.892144,2.0
6,exp_smoothing_fallback,monthly,0.01,multiplicative,14372.100115,21848.206915,2633.892144,2.0
7,exp_smoothing_fallback,monthly,0.01,additive,14372.100115,21848.206915,2633.892144,2.0
8,exp_smoothing_fallback,weekly,0.50,multiplicative,22010.222762,40311.500770,595.142083,1.0
9,exp_smoothing_fallback,weekly,0.50,additive,22010.222762,40311.500770,595.142083,1.0


,ds,y,yhat,level,entity
0,2025-09-01,7.000000,1424.877480,overall,all_businesses
1,2025-10-01,87.333333,1469.036602,overall,all_businesses
2,2025-11-01,160.142857,1513.195725,overall,all_businesses
3,2025-12-01,31743.030303,1557.354848,overall,all_businesses
4,2026-01-01,44660.730769,1601.513971,overall,all_businesses
5,2026-02-01,39012.769231,1645.673094,overall,all_businesses
6,2026-03-01,12756.846154,1689.832217,overall,all_businesses
7,2026-04-01,3728.922535,1733.991339,overall,all_businesses
8,2026-05-01,255.816568,1778.150462,overall,all_businesses
9,2025-08-01,151.500000,233.230424,sector,Cafes/Restaurants


Insight: selected model balances low error and stable behavior across aggregation levels.
